# Hosting Strands Agents with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. We will provide examples using Amazon Bedrock models and non-Bedrock models such as Azure OpenAI and Gemini.


### Tutorial Details


| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                        |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                     |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will  use a Strands Agent using Amazon Bedrock models

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using Strands Agents


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.

The architecture here will look as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="50%"/>
</div>

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models import BedrockModel

# Create a custom tool 
@tool
def weather(city:str):
    """Get weather information.

    Args:
        city: City for which weather will be returned

    Returns:
        Weather of provided city as a string.
    """
    print(city)
    if city.lower() == "athens":
        weather = "very sunny"
    else:
        weather = "sunny"
    return weather


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_bedrock(json.loads(args.payload))

#### Invoking local agent

In [ ]:
!python strands_claude.py '{"prompt": "What is the weather now in Athens ?"}'

## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def weather(city:str):
    """Get weather information.

    Args:
        city: City for which weather will be returned

    Returns:
        Weather of provided city as a string.
    """
    print(city)
    if city.lower() == "athens":
        weather = "very sunny"
    else:
        weather = "sunny"
    return weather


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

First we will use the AgentCore SDK and boto3 to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the SDK to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
import boto3
import json
import zipfile
import tempfile
import os
import time

boto_session = boto3.session.Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=region)

def create_or_get_execution_role(agent_name, region, account_id):
    """Create or retrieve an IAM execution role for the AgentCore runtime."""
    iam = boto3.client('iam')
    role_name = f"AgentCoreRuntime-{agent_name[:40]}"
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account_id},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/*"}
            }
        }]
    }
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": "*"},
            {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogDelivery", "logs:PutLogEvents",
                                            "logs:CreateLogStream", "logs:DescribeLogGroups", "logs:DescribeLogStreams"], "Resource": "*"}
        ]
    }
    try:
        role = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"Execution role for AgentCore runtime {agent_name}"
        )
        iam.put_role_policy(RoleName=role_name, PolicyName="AgentCoreRuntimePermissions",
                            PolicyDocument=json.dumps(permissions_policy))
        print(f"Created IAM role: {role_name}")
        time.sleep(10)  # Allow IAM propagation
        return role['Role']['Arn']
    except iam.exceptions.EntityAlreadyExistsException:
        arn = iam.get_role(RoleName=role_name)['Role']['Arn']
        print(f"Using existing IAM role: {role_name}")
        return arn

def package_and_upload_to_s3(agent_name, files, region, account_id):
    """Package agent code as a ZIP and upload to S3 for CodeZip deployment."""
    s3 = boto3.client('s3', region_name=region)
    bucket_name = f"bedrock-agentcore-{account_id}-{region}"
    try:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(Bucket=bucket_name,
                             CreateBucketConfiguration={'LocationConstraint': region})
        print(f"Created S3 bucket: {bucket_name}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"Using existing S3 bucket: {bucket_name}")

    with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmpf:
        zip_path = tmpf.name
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            if os.path.exists(f):
                zf.write(f, os.path.basename(f))
                print(f"  Packaged: {f}")
    s3_key = f"{agent_name}/deployment.zip"
    s3.upload_file(zip_path, bucket_name, s3_key)
    os.unlink(zip_path)
    print(f"Uploaded to s3://{bucket_name}/{s3_key}")
    return bucket_name, s3_key

agent_name = "strands_claude_getting_started"

# Create IAM execution role
role_arn = create_or_get_execution_role(agent_name, region, account_id)

# Package and upload agent code to S3
bucket_name, s3_key = package_and_upload_to_s3(
    agent_name,
    ["strands_claude.py", "requirements.txt"],
    region,
    account_id
)
print(f"\nAgent name:  {agent_name}")
print(f"Role ARN:    {role_arn}")
print(f"S3 package:  s3://{bucket_name}/{s3_key}")


### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
# Create the AgentCore Runtime using the CodeZip deployment type
create_response = agentcore_control.create_agent_runtime(
    agentRuntimeName=agent_name,
    agentRuntimeArtifact={
        'codeConfiguration': {
            'code': {'s3': {'bucket': bucket_name, 'prefix': s3_key}},
            'runtime': 'PYTHON_3_11',
            'entryPoint': ['strands_claude.py']
        }
    },
    roleArn=role_arn,
    networkConfiguration={'networkMode': 'PUBLIC'}
)
agent_runtime_id = create_response['agentRuntimeId']
agent_runtime_arn = create_response['agentRuntimeArn']
print(f"AgentCore Runtime created:")
print(f"  Runtime ID:  {agent_runtime_id}")
print(f"  Runtime ARN: {agent_runtime_arn}")


### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
import time
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
status = create_response.get('status', 'CREATING')
while status not in end_status:
    time.sleep(15)
    r = agentcore_control.get_agent_runtime(agentRuntimeId=agent_runtime_id)
    status = r['status']
    print(f"Status: {status}")
print(f"\nFinal status: {status}")


### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
import boto3
import json
from IPython.display import Markdown, display

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)

invoke_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How is the weather now in Athens ?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = invoke_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

try:
    events = []
    for event in invoke_response.get("response", []):
        events.append(event)
except Exception as e:
    events = [f"Error reading EventStream: {e}"]
response_text = json.loads(events[0].decode("utf-8"))
display(Markdown(response_text))


### Processing invocation results

We can now process our invocation results to include it in an application

In [ ]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = boto3_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

### Stopping a Session

You'll want to stop individual sessions when they're no longer needed.
This releases the microVM resources for that session while keeping the runtime alive
for new sessions. Below we demonstrate `stop_runtime_session`.

In [ ]:
# --- Inline Session Lifecycle Demo ---
# stop_runtime_session releases the microVM resources for this specific session while keeping the runtime alive for new sessions.


if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier='DEFAULT'
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

### Lifecycle Configuration Demo
Now let's demonstrate how to configure a runtime with a shorter idle timeout.
We'll create a second runtime with a 5-minute (300 second) idle timeout to show
how lifecycle configuration affects session behavior. Both runtimes will coexist.

In [ ]:
# --- Lifecycle Configuration Demo ---
# Create a second runtime with a 5-minute (300 second) idle timeout
# to show how lifecycle configuration affects session behavior.

agent_name_short = "strands_claude_short_timeout"

# Create IAM execution role for the short-timeout runtime
role_arn_short = create_or_get_execution_role(agent_name_short, region, account_id)

# Package and upload (reuses existing S3 bucket)
bucket_name_short, s3_key_short = package_and_upload_to_s3(
    agent_name_short,
    ["strands_claude.py", "requirements.txt"],
    region,
    account_id
)

# Create the second AgentCore Runtime
create_response_short = agentcore_control.create_agent_runtime(
    agentRuntimeName=agent_name_short,
    agentRuntimeArtifact={
        'codeConfiguration': {
            'code': {'s3': {'bucket': bucket_name_short, 'prefix': s3_key_short}},
            'runtime': 'PYTHON_3_11',
            'entryPoint': ['strands_claude.py']
        }
    },
    roleArn=role_arn_short,
    networkConfiguration={'networkMode': 'PUBLIC'}
)
agent_runtime_id_short = create_response_short['agentRuntimeId']
agent_runtime_arn_short = create_response_short['agentRuntimeArn']
print(f"Second runtime launched: {agent_runtime_id_short}")

# Wait for second runtime to be ready
status_short = create_response_short.get('status', 'CREATING')
while status_short not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    time.sleep(15)
    r = agentcore_control.get_agent_runtime(agentRuntimeId=agent_runtime_id_short)
    status_short = r['status']
    print(f"Short timeout runtime status: {status_short}")

# Update the second runtime with a 5-minute idle timeout
# UpdateAgentRuntime is a full-replacement API — re-supply all required fields
current_runtime = agentcore_control.get_agent_runtime(agentRuntimeId=agent_runtime_id_short)
update_response = agentcore_control.update_agent_runtime(
    agentRuntimeId=agent_runtime_id_short,
    agentRuntimeArtifact=current_runtime['agentRuntimeArtifact'],
    roleArn=current_runtime['roleArn'],
    networkConfiguration=current_runtime['networkConfiguration'],
    lifecycleConfiguration={'idleRuntimeSessionTimeout': 300}  # 5 minutes
)
print(f"Second runtime updated with 5-minute idle timeout")

# Invoke the second runtime to verify it works
invoke_short = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn_short,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 3+3?"})
)
short_events = [e for e in invoke_short.get("response", [])]
if short_events:
    print(f"Second runtime response: {json.loads(short_events[0].decode('utf-8'))}")


## Session Lifecycle Best Practices

AgentCore Runtime costs are based on vCPU and Memory. A best practice to avoid undesired costs is to explicitly stop the session or set up a properly configured idle timeout, so the session will be terminated.

To manage costs effectively:

- **Configure idle timeout**: Set an appropriate idle timeout during session creation to automatically stop inactive sessions. Choose a value based on your use case (e.g., shorter for development/testing, longer for production workloads).
- **Stop sessions when done**: Use `stop_runtime_session` to release the microVM resources for a specific session while keeping the runtime alive for new sessions.

## Cleanup

Let's now clean up the AgentCore Runtime and associated resources. We delete the runtime first to avoid undesired costs, then clean up supporting resources like ECR repositories.

In [ ]:
# Runtime info
print(f"Agent Runtime ID:  {agent_runtime_id}")
print(f"Agent Runtime ARN: {agent_runtime_arn}")


In [ ]:
# --- Cleanup: Delete runtimes and S3 artifacts ---
import boto3

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
s3_client = boto3.client('s3', region_name=region)

# Stop the active session to release microVM resources
if 'runtime_session_id' in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=agent_runtime_arn,
            runtimeSessionId=runtime_session_id,
            qualifier='DEFAULT'
        )
        print(f"Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"Could not stop session: {e}")

# Delete original runtime
try:
    agentcore_control.delete_agent_runtime(agentRuntimeId=agent_runtime_id)
    print(f"Runtime '{agent_name}' deleted")
except Exception as e:
    print(f"Could not delete runtime: {e}")

# Delete short-timeout runtime
if 'agent_runtime_id_short' in dir():
    try:
        agentcore_control.delete_agent_runtime(agentRuntimeId=agent_runtime_id_short)
        print(f"Short-timeout runtime deleted")
    except Exception as e:
        print(f"Could not delete short-timeout runtime: {e}")

# Delete S3 deployment artifacts
try:
    s3_client.delete_object(Bucket=bucket_name, Key=s3_key)
    print(f"S3 artifact deleted: s3://{bucket_name}/{s3_key}")
except Exception as e:
    print(f"Could not delete S3 artifact: {e}")

# Delete local config file if it exists
import subprocess
subprocess.run(["rm", "-f", ".bedrock_agentcore.yaml"], capture_output=True)
print("Cleanup complete.")


# Congratulations!